# TMDB Movie Data Analysis — Data Cleaning and Preprocessing

## 1. Introduction

This notebook performs data cleaning and preprocessing on the raw movie dataset extracted from The Movie Database (TMDb).

Raw API data can contain nested JSON structures, missing values, incorrect data types, duplicate records, and unrealistic numeric values. These issues must be addressed before conducting reliable exploratory data analysis and calculating movie performance KPIs.

### Objectives
- Inspect the raw dataset and identify data quality issues.
- Remove irrelevant columns.
- Extract useful information from nested JSON fields.
- Correct data types and handle missing or unrealistic values.
- Remove duplicate and invalid records.
- Retain released movies with sufficient information.
- Reorder columns and save the cleaned dataset.

The original extracted data will remain unchanged so the cleaning process can be reproduced.

In [4]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

raw_file = PROJECT_ROOT / "data" / "raw" / "tmdb_movies_raw.json"

if not raw_file.exists():
    raise FileNotFoundError(
        f"Raw dataset not found: {raw_file}. "
        "Run Notebook 01 first."
    )

df = pd.read_json(raw_file, orient="records")

print("Raw dataset loaded successfully.")
print("Shape:", df.shape)

Raw dataset loaded successfully.
Shape: (18, 31)


In [5]:
df.iloc[3]

adult                                                                False
backdrop_path                             /mDfJG3LC3Dqb67AZ52x3Z0jU0uB.jpg
belongs_to_collection    {'id': 86311, 'name': 'The Avengers Collection...
budget                                                           300000000
genres                   [{'id': 12, 'name': 'Adventure'}, {'id': 28, '...
homepage                 https://www.marvel.com/movies/avengers-infinit...
id                                                                  299536
imdb_id                                                          tt4154756
origin_country                                                        [US]
original_language                                                       en
original_title                                      Avengers: Infinity War
overview                 As the Avengers and their allies have continue...
popularity                                                        142.7426
poster_path              

## 2. Initial Data Inspection

Before modifying the data, we inspect its structure and quality.

We will examine the dimensions, column names, data types, missing-value counts, duplicate records, and a sample of the dataset. This provides a baseline for evaluating the effects of the cleaning operations.

In [6]:
print("Dataset dimensions:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nMissing values:")
display(
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("Missing Count")
)

print("\nDuplicate rows:", df.duplicated(subset="title").sum())

display(df.head(3))

Dataset dimensions: (18, 31)

Column names:
['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count', 'cast', 'cast_size', 'director', 'crew_size']

Data types:


,Data Type
adult,bool
backdrop_path,str
belongs_to_collection,object
budget,int64
genres,object
homepage,str
id,int64
imdb_id,str
origin_country,object
original_language,str



Missing values:


,Missing Count
belongs_to_collection,2
adult,0
backdrop_path,0
budget,0
genres,0
homepage,0
id,0
imdb_id,0
origin_country,0
original_language,0



Duplicate rows: 0


,adult,backdrop_path,belongs_to_collection,budget,genres,homepage,id,imdb_id,origin_country,original_language,...,status,tagline,title,video,vote_average,vote_count,cast,cast_size,director,crew_size
0,False,/7RyHsO4yDXtBv1zUU3mTpHeQ0d5.jpg,"{'id': 86311, 'name': 'The Avengers Collection...",356000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 878, ...",https://www.marvel.com/movies/avengers-endgame,299534,tt4154796,[US],en,...,Released,Avenge the fallen.,Avengers: Endgame,False,8.200,29005,"[Robert Downey Jr., Chris Evans, Mark Ruffalo,...",103,Anthony Russo | Joe Russo,613
1,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,"{'id': 87096, 'name': 'Avatar Collection', 'po...",237000000,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...",https://www.avatar.com/movies/avatar,19995,tt0499549,[US],en,...,Released,Enter the world of Pandora.,Avatar,False,7.611,34845,"[Sam Worthington, Zoe Saldaña, Sigourney Weave...",68,James Cameron,999
2,False,/8BTsTfln4jlQrLXUBquXJ0ASQy9.jpg,"{'id': 10, 'name': 'Star Wars Collection', 'po...",245000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",http://www.starwars.com/films/star-wars-episod...,140607,tt2488496,[US],en,...,Released,Every generation has a story.,Star Wars: The Force Awakens,False,7.200,21013,"[Daisy Ridley, John Boyega, Oscar Isaac, Adam ...",99,J.J. Abrams,270


## 3. Remove Irrelevant Columns

Some fields are not required for the planned analysis. We will remove `adult`, `imdb_id`, `original_title`, `video`, and `homepage` when they are present.

The `errors="ignore"` option allows the code to work even if a column is absent from the API response.

In [7]:
irrelevant_columns = [
    "adult",
    "imdb_id",
    "original_title",
    "video",
    "homepage"
]

df = df.drop(
    columns=irrelevant_columns,
    errors="ignore"
)

print("Shape after dropping irrelevant columns:", df.shape)

Shape after dropping irrelevant columns: (18, 26)


## 4. Extract Information from Nested JSON Columns

TMDb returns several attributes as lists of dictionaries rather than simple text values.

For example, a movie's genres may look like:

`[{"id": 28, "name": "Action"}, {"id": 878, "name": "Science Fiction"}]`

For analysis, we need readable values rather than nested structures. We will extract:

- `belongs_to_collection`: collection name.
- `genres`: genre names separated by `|`.
- `spoken_languages`: language names separated by `|`.
- `production_countries`: country names separated by `|`.
- `production_companies`: company names separated by `|`.

We will use reusable helper functions within this notebook to handle missing values and unexpected data structures consistently.

In [8]:
def extract_names(value, key="name"):
    """Extract names from a list of dictionaries."""
    if isinstance(value, list):
        names = [
            item.get(key)
            for item in value
            if isinstance(item, dict) and item.get(key)
        ]
        return " | ".join(names) if names else np.nan

    return np.nan


def extract_collection(value):
    """Extract a collection name from a dictionary."""
    if isinstance(value, dict):
        return value.get("name") or np.nan

    return np.nan

In [9]:
if "belongs_to_collection" in df.columns:
    df["belongs_to_collection"] = (
        df["belongs_to_collection"].apply(extract_collection)
    )

list_columns = {
    "genres": "name",
    "spoken_languages": "english_name",
    "production_countries": "name",
    "production_companies": "name"
}

for column, key in list_columns.items():
    if column in df.columns:
        df[column] = df[column].apply(
            lambda value: extract_names(value, key)
        )

print("Nested columns processed.")
display(df.head(3))

Nested columns processed.


,backdrop_path,belongs_to_collection,budget,genres,id,origin_country,original_language,overview,popularity,poster_path,...,spoken_languages,status,tagline,title,vote_average,vote_count,cast,cast_size,director,crew_size
0,/7RyHsO4yDXtBv1zUU3mTpHeQ0d5.jpg,The Avengers Collection,356000000,Adventure | Science Fiction | Action,299534,[US],en,After the devastating events of Avengers: Infi...,87.1502,/bR8ISy1O9XQxqiy0fQFw2BX72RQ.jpg,...,English | Japanese | Xhosa,Released,Avenge the fallen.,Avengers: Endgame,8.200,29005,"[Robert Downey Jr., Chris Evans, Mark Ruffalo,...",103,Anthony Russo | Joe Russo,613
1,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,Avatar Collection,237000000,Science Fiction | Action | Adventure,19995,[US],en,"In the 22nd century, a paraplegic Marine is di...",48.3380,/gKY6q7SjCkAU6FqvqWybDYgUKIF.jpg,...,English | Spanish,Released,Enter the world of Pandora.,Avatar,7.611,34845,"[Sam Worthington, Zoe Saldaña, Sigourney Weave...",68,James Cameron,999
2,/8BTsTfln4jlQrLXUBquXJ0ASQy9.jpg,Star Wars Collection,245000000,Adventure | Action | Science Fiction,140607,[US],en,Thirty years after defeating the Galactic Empi...,45.2373,/wqnLdwVXoBjKibFRR5U3y0aDUhs.jpg,...,English,Released,Every generation has a story.,Star Wars: The Force Awakens,7.200,21013,"[Daisy Ridley, John Boyega, Oscar Isaac, Adam ...",99,J.J. Abrams,270


## 5. Inspect Extracted Categorical Values

After transforming the nested fields, we use `value_counts()` to inspect the most frequent genres, production countries, spoken languages, companies, and collections.

This helps identify unexpected values, empty fields, inconsistent labels, or extraction problems. Because some columns contain multiple values separated by `|`, we will split them before counting individual categories.

In [10]:
categorical_columns = [
    "genres",
    "spoken_languages",
    "production_countries",
    "production_companies",
    "belongs_to_collection"
]

for column in categorical_columns:
    if column not in df.columns:
        continue

    print(f"\n--- {column} ---")

    if column == "belongs_to_collection":
        display(df[column].value_counts(dropna=False).head(10))
    else:
        counts = (
            df[column]
            .dropna()
            .str.split(" | ", regex=False)
            .explode()
            .value_counts()
        )
        display(counts.head(10))


--- genres ---


genres
Adventure          15
Science Fiction    10
Action             10
Family              5
Animation           4
Fantasy             4
Thriller            3
Drama               2
Romance             2
Crime               1
Name: count, dtype: int64


--- spoken_languages ---


spoken_languages
English     18
Xhosa        3
Russian      3
Spanish      2
French       2
Japanese     1
German       1
Swedish      1
Italian      1
Hindi        1
Name: count, dtype: int64


--- production_countries ---


production_countries
United States of America    18
United Kingdom               2
Name: count, dtype: int64


--- production_companies ---


production_companies
Marvel Studios                   5
Universal Pictures               3
Lightstorm Entertainment         2
20th Century Fox                 2
Lucasfilm Ltd.                   2
Amblin Entertainment             2
Walt Disney Pictures             2
Walt Disney Animation Studios    2
Dune Entertainment               1
Ingenious Film Partners          1
Name: count, dtype: int64


--- belongs_to_collection ---


belongs_to_collection
The Avengers Collection                4
Star Wars Collection                   2
NaN                                    2
Jurassic Park Collection               2
Frozen Collection                      2
Avatar Collection                      1
The Lion King (Reboot) Collection      1
The Fast and the Furious Collection    1
Black Panther Collection               1
Harry Potter Collection                1
Name: count, dtype: int64

## 6. Correct Data Types

Data types affect how Pandas performs calculations, comparisons, and filtering.

We will convert the movie identifier, budget, revenue, popularity, vote count, vote average, and runtime to numeric types where available. Invalid values will become `NaN` instead of causing the entire conversion to fail.

The release date will be converted to a datetime type, allowing us to derive release years and analyze trends later.

In [11]:
numeric_columns = [
    "id",
    "budget",
    "revenue",
    "popularity",
    "vote_count",
    "vote_average",
    "runtime",
    "cast_size",
    "crew_size"
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )

if "release_date" in df.columns:
    df["release_date"] = pd.to_datetime(
        df["release_date"],
        errors="coerce"
    )

print("Data types converted.")
display(df.dtypes.to_frame("Data Type"))

Data types converted.


,Data Type
backdrop_path,str
belongs_to_collection,str
budget,int64
genres,str
id,int64
origin_country,object
original_language,str
overview,str
popularity,float64
poster_path,str


## 7. Handle Missing and Unrealistic Values

Some movie records use zero to indicate that budget, revenue, or runtime is unavailable. Zero can distort financial calculations, so we will replace those values with `NaN`.

We will also:
- Convert budget and revenue into millions of USD.
- Treat a rating as unavailable when the movie has no votes.
- Replace known placeholder text in overview and tagline fields.
- Preserve other missing values for later analysis rather than inventing data.

We will not infer financial figures from similar movies because unsupported estimates could distort the results.

In [12]:
for column in ["budget", "revenue", "runtime"]:
    if column in df.columns:
        df[column] = df[column].mask(df[column] == 0, np.nan)

if "vote_count" in df.columns and "vote_average" in df.columns:
    df.loc[df["vote_count"] == 0, "vote_average"] = np.nan

if "budget" in df.columns:
    df["budget_musd"] = df["budget"] / 1_000_000

if "revenue" in df.columns:
    df["revenue_musd"] = df["revenue"] / 1_000_000

print("Numeric values cleaned and financial values converted to millions.")

Numeric values cleaned and financial values converted to millions.


In [13]:
placeholder_values = [
    "",
    "No Data",
    "No data",
    "N/A",
    "n/a",
    "None",
    "null",
    "Unknown"
]

for column in ["overview", "tagline"]:
    if column in df.columns:
        text_values = df[column].astype("string").str.strip()

        df[column] = text_values.mask(
            text_values.isin(placeholder_values)
        )

print("Text placeholders cleaned.")

Text placeholders cleaned.


## 8. Remove Invalid and Duplicate Records

Duplicate records can bias rankings and summary statistics. We will first remove exact duplicate rows, then remove duplicate movie IDs while retaining the first occurrence.

Rows without a valid movie ID or title will also be removed because they cannot be reliably identified in the final analysis.

We will preserve the cleaning sequence and record the number of remaining rows after each operation.

In [14]:
rows_before = len(df)

# Exact duplicate rows (string copy avoids the unhashable list/dict error)
df = df[~df.astype(str).duplicated()]

if "id" in df.columns:
    df = df.drop_duplicates(subset="id", keep="first")
    df = df[df["id"].notna() & (df["id"] > 0)]

if "title" in df.columns:
    df = df[
        df["title"].notna()
        & df["title"].astype("string").str.strip().ne("")
    ]

print("Rows before:", rows_before)
print("Rows after:", len(df))
print("Rows removed:", rows_before - len(df))

Rows before: 18
Rows after: 18
Rows removed: 0


## 9. Filter Records by Data Completeness

The project requires retaining rows with at least 10 non-missing values.

We will use `count(axis=1)` to count non-null values in each row and retain only records that meet this threshold. This is a general completeness rule, not a guarantee that every remaining record contains every field needed for every KPI.

In [15]:
non_null_counts = df.notna().sum(axis=1)

print("Rows before completeness filtering:", len(df))

df = df.loc[non_null_counts >= 10].copy()

print("Rows after completeness filtering:", len(df))

Rows before completeness filtering: 18
Rows after completeness filtering: 18


## 10. Filter Released Movies

The project focuses on movies whose TMDb status is `Released`.

We will filter using the status value and then remove the `status` column because it is no longer needed for the analysis.

This filtering step may exclude otherwise valid movies with different statuses, which is intentional under the project requirements.

In [16]:
if "status" in df.columns:
    print("Statuses before filtering:")
    display(df["status"].value_counts(dropna=False))

    df = df.loc[df["status"].eq("Released")].copy()
    df = df.drop(columns="status")

print("Rows after released-movie filtering:", len(df))

Statuses before filtering:


status
Released    18
Name: count, dtype: int64

Rows after released-movie filtering: 18


## 11. Reorder Columns and Finalize the Dataset

The final DataFrame will follow the column order specified in the project requirements.

Some requested columns may be missing if they were not returned by the API. We will check for those columns before reordering. Missing requested fields will be added as `NaN` so the dataset has a consistent schema.

Finally, we reset the index and save the cleaned dataset to the processed data directory.

In [17]:
final_columns = [
    "id",
    "title",
    "tagline",
    "release_date",
    "genres",
    "belongs_to_collection",
    "original_language",
    "budget_musd",
    "revenue_musd",
    "production_companies",
    "production_countries",
    "vote_count",
    "vote_average",
    "popularity",
    "runtime",
    "overview",
    "spoken_languages",
    "poster_path",
    "cast",
    "cast_size",
    "director",
    "crew_size"
]

for column in final_columns:
    if column not in df.columns:
        df[column] = np.nan

df = df[final_columns].reset_index(drop=True)

print("Final dataset shape:", df.shape)
display(df.head())

Final dataset shape: (18, 22)


,id,title,tagline,release_date,genres,belongs_to_collection,original_language,budget_musd,revenue_musd,production_companies,...,vote_average,popularity,runtime,overview,spoken_languages,poster_path,cast,cast_size,director,crew_size
0,299534,Avengers: Endgame,Avenge the fallen.,2019-04-24,Adventure | Science Fiction | Action,The Avengers Collection,en,356.0,2927.449455,Marvel Studios,...,8.200,87.1502,181,After the devastating events of Avengers: Infi...,English | Japanese | Xhosa,/bR8ISy1O9XQxqiy0fQFw2BX72RQ.jpg,"[Robert Downey Jr., Chris Evans, Mark Ruffalo,...",103,Anthony Russo | Joe Russo,613
1,19995,Avatar,Enter the world of Pandora.,2009-12-16,Science Fiction | Action | Adventure,Avatar Collection,en,237.0,2923.706026,Dune Entertainment | Lightstorm Entertainment ...,...,7.611,48.3380,162,"In the 22nd century, a paraplegic Marine is di...",English | Spanish,/gKY6q7SjCkAU6FqvqWybDYgUKIF.jpg,"[Sam Worthington, Zoe Saldaña, Sigourney Weave...",68,James Cameron,999
2,140607,Star Wars: The Force Awakens,Every generation has a story.,2015-12-15,Adventure | Action | Science Fiction,Star Wars Collection,en,245.0,2068.223624,Lucasfilm Ltd. | Bad Robot,...,7.200,45.2373,136,Thirty years after defeating the Galactic Empi...,English,/wqnLdwVXoBjKibFRR5U3y0aDUhs.jpg,"[Daisy Ridley, John Boyega, Oscar Isaac, Adam ...",99,J.J. Abrams,270
3,299536,Avengers: Infinity War,An entire universe. Once and for all.,2018-04-25,Adventure | Action | Science Fiction,The Avengers Collection,en,300.0,2052.415039,Marvel Studios,...,8.245,142.7426,149,As the Avengers and their allies have continue...,English | Xhosa,/7WsyChQLEftFiDOVTGkv3hFpyyt.jpg,"[Robert Downey Jr., Chris Evans, Chris Hemswor...",69,Joe Russo | Anthony Russo,738
4,597,Titanic,Nothing on earth could come between them.,1997-12-18,Drama | Romance,NaN,en,200.0,2264.162353,Paramount Pictures | 20th Century Fox | Lights...,...,7.901,65.2492,194,101-year-old Rose DeWitt Bukater tells the sto...,English | French | German | Swedish | Italian ...,/9xjZS2rlVxm8SFx8kPC3aIGCOYQ.jpg,"[Leonardo DiCaprio, Kate Winslet, Billy Zane, ...",116,James Cameron,271


In [18]:
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

output_file = processed_dir / "tmdb_movies_cleaned.csv"

df.to_csv(output_file, index=False)

print("Cleaned dataset saved to:", output_file)

Cleaned dataset saved to: d:\AmaliTech\TMDB-Movie-Analysis\data\processed\tmdb_movies_cleaned.csv


## 12. Final Data Quality Checks

We will perform a final inspection to confirm that the cleaned dataset has the expected schema, no duplicate IDs, valid titles, and only released movies.

Because the `status` column has been removed, the released-movie condition was enforced in the previous step. The following checks focus on the final structure and remaining data quality issues.

In [19]:
print("Final shape:", df.shape)
print("Duplicate IDs:", df["id"].duplicated().sum())
print("Missing IDs:", df["id"].isna().sum())
print("Missing titles:", df["title"].isna().sum())

print("\nMissing values per column:")
display(
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("Missing Count")
)

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

Final shape: (18, 22)
Duplicate IDs: 0
Missing IDs: 0
Missing titles: 0

Missing values per column:


,Missing Count
belongs_to_collection,2
id,0
title,0
tagline,0
release_date,0
genres,0
original_language,0
budget_musd,0
revenue_musd,0
production_companies,0



Data types:


,Data Type
id,int64
title,str
tagline,string
release_date,datetime64[us]
genres,str
belongs_to_collection,str
original_language,str
budget_musd,float64
revenue_musd,float64
production_companies,str
